In [2]:
from openpyxl import load_workbook
wb=load_workbook("test.xlsx")
ws=wb.active
table_range=ws.tables['Table1']
table_range.ref

'A1:C32'

In [3]:
import pandas as pd

location_all = pd.read_excel("test.xlsx",sheet_name='Sheet1')
location_all[['lat','lon']]=location_all['Address'].str.split(',',expand=True)
location_all['lat']=pd.to_numeric(location_all['lat'],errors='coerce')
location_all['lon']=pd.to_numeric(location_all['lon'],errors='coerce')
location_all

,Day,Name,Address,lat,lon
0,5/19/2022,Vịt 34 Mỹ Đình,"21.03030065556449, 105.7743223558026",21.030301,105.774322
1,5/20/2022,Sakuko Store Hàm Nghi,"21.035866644604525, 105.7624111282254",21.035867,105.762411
2,5/21/2022,Decathlon Royal City,"21.001291158876537, 105.81660960626195",21.001291,105.816610
3,5/21/2022,Nét Huế Royal City,"21.001389568565166, 105.81647141689129",21.001390,105.816471
4,5/21/2022,Bowling Royal City,"21.00202081326225, 105.81667281053721",21.002021,105.816673
5,5/21/2022,Trượt băng Royal City,"21.00289376577585, 105.81554470187945",21.002894,105.815545
6,5/22/2022,Tasco,"21.015690901830617, 105.78289651483884",21.015691,105.782897
7,5/24/2022,4P Keangnam,"21.01726960347157, 105.78366481439532",21.017270,105.783665
8,5/24/2022,Starbuck Keangnam,"21.016470924882796, 105.78392215515419",21.016471,105.783922
9,5/27/2022,Tasco,"21.015690901830617, 105.78289651483884",21.015691,105.782897


In [7]:
df=location_all[location_all['Day']=='5/21/2022']
df['lat_lag'] = df.groupby('Day')['lat'].shift(1)
df['lon_lag'] = df.groupby('Day')['lon'].shift(1)
df['row_num'] = df.groupby('Day').cumcount()+1
df

C:\Users\thanh\AppData\Local\Temp\ipykernel_7100\942024941.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['lat_lag'] = df.groupby('Day')['lat'].shift(1)
C:\Users\thanh\AppData\Local\Temp\ipykernel_7100\942024941.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['lon_lag'] = df.groupby('Day')['lon'].shift(1)
C:\Users\thanh\AppData\Local\Temp\ipykernel_7100\942024941.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_in

,Day,Name,Address,lat,lon,lat_lag,lon_lag,row_num
2,5/21/2022,Decathlon Royal City,"21.001291158876537, 105.81660960626195",21.001291,105.816610,NaN,NaN,1
3,5/21/2022,Nét Huế Royal City,"21.001389568565166, 105.81647141689129",21.001390,105.816471,21.001291,105.816610,2
4,5/21/2022,Bowling Royal City,"21.00202081326225, 105.81667281053721",21.002021,105.816673,21.001390,105.816471,3
5,5/21/2022,Trượt băng Royal City,"21.00289376577585, 105.81554470187945",21.002894,105.815545,21.002021,105.816673,4


In [8]:
import openrouteservice
from openrouteservice import convert
import folium
from folium.plugins import MarkerCluster
import json

client = openrouteservice.Client(key='5b3ce3597851110001cf6248b5df373961814230b56d53ceb4c83d66')

m = folium.Map(
                location=[20.98507688055118, 105.84230817298926],
                zoom_start=7, 
                control_scale=True,
                tiles="cartodbpositron"
               )

# if the points are too close to each other, cluster them, create a cluster overlay with MarkerCluster
marker_cluster = MarkerCluster().add_to(m)

for i,r in df.iterrows():
    if r['row_num'] != 1: 
        coords = ((r['lon'], r['lat']),(r['lon_lag'], r['lat_lag'])) # Exchange Lat and Lon in database
        res = client.directions(coords)
        geometry = client.directions(coords)['routes'][0]['geometry']
        decoded = convert.decode_polyline(geometry)

        distance_txt = "<h4> <b>Distance :&nbsp" + "<strong>"+str(round(res['routes'][0]['summary']['distance']/1000,1))+" Km </strong>" +"</h4></b>"
        duration_txt = "<h4> <b>Duration :&nbsp" + "<strong>"+str(round(res['routes'][0]['summary']['duration']/60,1))+" Mins. </strong>" +"</h4></b>"

        folium.GeoJson(decoded).add_child(folium.Popup(distance_txt+duration_txt,max_width=300)).add_to(m)

    location = (r["lat"], r["lon"])
    folium.Marker(location=location,
                      popup = r['Name'],
                      tooltip=r['Name'])\
    .add_to(marker_cluster)    

m